# 第 2 週實作：資料視覺化與 EDA（互動圖表）
# Week 2 Lab: Data Visualization & EDA (Interactive Charts)

## 學習目標 Learning Objectives
1. 掌握 Matplotlib 進階繪圖（子圖佈局、樣式自訂）
2. 使用 Seaborn 進行統計視覺化（pairplot, heatmap, violinplot）
3. 利用 Plotly 製作互動式圖表（散佈圖、3D、動畫）
4. 實踐完整的 EDA 流程（描述統計、分布檢查、相關性分析、異常值偵測）
5. 學會缺失值處理策略

---

## 0. 環境設定與資料載入 Environment Setup & Data Loading

In [ ]:
# 環境設定 Environment Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')

# Matplotlib 中文字體設定（Windows）
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

# Seaborn 全域設定
sns.set_theme(style='whitegrid', font_scale=1.1)

# 版本確認
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Matplotlib: {plt.matplotlib.__version__}")
print(f"Seaborn: {sns.__version__}")
print(f"環境設定完成！ Environment ready!")

In [ ]:
# 載入資料集 Load Datasets

# Titanic 資料集 — 經典的分類問題資料
titanic = sns.load_dataset('titanic')
print("=== Titanic 資料集 ===")
print(f"形狀 Shape: {titanic.shape}")
print(f"欄位 Columns: {list(titanic.columns)}")
print()

# Iris 資料集 — 經典的多分類問題資料
iris = sns.load_dataset('iris')
print("=== Iris 資料集 ===")
print(f"形狀 Shape: {iris.shape}")
print(f"欄位 Columns: {list(iris.columns)}")
print()

# 預覽 Titanic 前 5 筆
titanic.head()

---

## 1. Matplotlib 進階繪圖 Advanced Matplotlib

### 1.1 基本子圖佈局 Basic Subplot Layout

使用 `plt.subplots()` 建立多子圖佈局，這是 Matplotlib 物件導向 (Object-Oriented) API 的核心。

In [ ]:
# 2x2 子圖佈局：展示 Titanic 資料的四個面向
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Titanic 資料集多面向探索 Multi-Aspect Exploration', fontsize=16, fontweight='bold')

# 子圖 1：年齡分布 Age Distribution
axes[0, 0].hist(titanic['age'].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.7)
axes[0, 0].axvline(titanic['age'].mean(), color='red', linestyle='--', label=f"Mean={titanic['age'].mean():.1f}")
axes[0, 0].axvline(titanic['age'].median(), color='green', linestyle=':', label=f"Median={titanic['age'].median():.1f}")
axes[0, 0].set_title('年齡分布 Age Distribution')
axes[0, 0].set_xlabel('年齡 Age')
axes[0, 0].set_ylabel('人數 Count')
axes[0, 0].legend()

# 子圖 2：各艙等人數 Passengers by Class
class_counts = titanic['pclass'].value_counts().sort_index()
colors = ['#2196F3', '#FF9800', '#4CAF50']
axes[0, 1].bar(class_counts.index.astype(str), class_counts.values, color=colors, edgecolor='white')
axes[0, 1].set_title('各艙等乘客人數 Passengers by Class')
axes[0, 1].set_xlabel('艙等 Class')
axes[0, 1].set_ylabel('人數 Count')
for i, v in enumerate(class_counts.values):
    axes[0, 1].text(i, v + 5, str(v), ha='center', fontweight='bold')

# 子圖 3：票價散佈圖 Fare Scatter
survived_colors = titanic['survived'].map({0: '#E53935', 1: '#43A047'})
axes[1, 0].scatter(titanic['age'], titanic['fare'], c=survived_colors, alpha=0.5, s=20)
axes[1, 0].set_title('年齡 vs 票價（紅=罹難, 綠=存活）')
axes[1, 0].set_xlabel('年齡 Age')
axes[1, 0].set_ylabel('票價 Fare')
axes[1, 0].set_ylim(0, 300)  # 限制 Y 軸避免極端值影響

# 子圖 4：存活率圓餅圖 Survival Pie Chart
survived_counts = titanic['survived'].value_counts()
axes[1, 1].pie(survived_counts, labels=['罹難 Died', '存活 Survived'],
               autopct='%1.1f%%', colors=['#E53935', '#43A047'],
               startangle=90, explode=[0, 0.05])
axes[1, 1].set_title('存活比例 Survival Rate')

plt.tight_layout()
plt.show()

### 1.2 GridSpec 不等大小子圖 Unequal Subplot Sizes

GridSpec 讓我們能建立不等大小的子圖佈局，適合建立 EDA 儀表板。

In [ ]:
# 使用 GridSpec 建立 EDA 儀表板風格佈局
fig = plt.figure(figsize=(16, 10))
gs = GridSpec(2, 3, figure=fig, hspace=0.3, wspace=0.3)

# 主圖：佔左上 2x2 區域 — 年齡 vs 票價散佈圖
ax_main = fig.add_subplot(gs[0, :2])
for survived, group in titanic.groupby('survived'):
    label = '存活 Survived' if survived == 1 else '罹難 Died'
    color = '#43A047' if survived == 1 else '#E53935'
    ax_main.scatter(group['age'], group['fare'], alpha=0.5, s=30, label=label, color=color)
ax_main.set_title('年齡 vs 票價（依存活狀態）Age vs Fare by Survival', fontsize=13)
ax_main.set_xlabel('年齡 Age')
ax_main.set_ylabel('票價 Fare')
ax_main.set_ylim(0, 300)
ax_main.legend()

# 右上：年齡直方圖
ax_right = fig.add_subplot(gs[0, 2])
ax_right.hist(titanic['age'].dropna(), bins=25, orientation='horizontal',
              color='steelblue', edgecolor='white', alpha=0.7)
ax_right.set_title('年齡分布')
ax_right.set_xlabel('人數 Count')
ax_right.set_ylabel('年齡 Age')

# 下方左：各艙等存活率
ax_bottom_left = fig.add_subplot(gs[1, 0])
survival_by_class = titanic.groupby('pclass')['survived'].mean()
ax_bottom_left.bar(survival_by_class.index.astype(str), survival_by_class.values,
                   color=['#2196F3', '#FF9800', '#4CAF50'])
ax_bottom_left.set_title('各艙等存活率 Survival by Class')
ax_bottom_left.set_ylabel('存活率 Survival Rate')
for i, v in enumerate(survival_by_class.values):
    ax_bottom_left.text(i, v + 0.01, f'{v:.1%}', ha='center', fontweight='bold')

# 下方中：性別存活率
ax_bottom_mid = fig.add_subplot(gs[1, 1])
survival_by_sex = titanic.groupby('sex')['survived'].mean()
ax_bottom_mid.bar(survival_by_sex.index, survival_by_sex.values,
                  color=['#E91E63', '#1565C0'])
ax_bottom_mid.set_title('性別存活率 Survival by Gender')
ax_bottom_mid.set_ylabel('存活率 Survival Rate')
for i, v in enumerate(survival_by_sex.values):
    ax_bottom_mid.text(i, v + 0.01, f'{v:.1%}', ha='center', fontweight='bold')

# 下方右：票價分布（箱型圖）
ax_bottom_right = fig.add_subplot(gs[1, 2])
titanic.boxplot(column='fare', by='pclass', ax=ax_bottom_right)
ax_bottom_right.set_title('票價分布（依艙等）Fare by Class')
ax_bottom_right.set_xlabel('艙等 Class')
ax_bottom_right.set_ylabel('票價 Fare')
ax_bottom_right.set_ylim(0, 200)
fig.suptitle('')  # 移除 boxplot 自動產生的標題

plt.show()

### 1.3 樣式切換 Style Switching

Matplotlib 內建多種樣式表，可快速改變圖表風格。

In [ ]:
# 展示四種不同樣式
styles = ['default', 'seaborn-v0_8-whitegrid', 'ggplot', 'dark_background']
x = np.linspace(0, 2 * np.pi, 100)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, style_name in zip(axes.flat, styles):
    with plt.style.context(style_name):
        # 注意：context 會影響新建的 figure，但這裡我們直接畫在現有 axes 上
        ax.plot(x, np.sin(x), label='sin(x)', linewidth=2)
        ax.plot(x, np.cos(x), label='cos(x)', linewidth=2)
        ax.fill_between(x, np.sin(x), alpha=0.2)
        ax.set_title(f'Style: {style_name}', fontsize=12)
        ax.legend()
        ax.set_xlabel('x')
        ax.set_ylabel('y')

plt.suptitle('Matplotlib 四種內建樣式比較 Built-in Style Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 重設為預設主題
sns.set_theme(style='whitegrid', font_scale=1.1)

---

## 2. Seaborn 統計視覺化 Seaborn Statistical Visualization

Seaborn 是建構在 Matplotlib 之上的統計視覺化函式庫，能用簡潔的語法產出專業的統計圖表。

### 2.1 pairplot — 成對關係圖

In [ ]:
# Iris 資料集的 pairplot
# 對角線 (diagonal)：各物種的 KDE 密度圖
# 非對角線 (off-diagonal)：兩兩變數的散佈圖
g = sns.pairplot(iris, hue='species', diag_kind='kde',
                 plot_kws={'alpha': 0.6, 's': 40},
                 palette='husl', height=2.5)
g.fig.suptitle('Iris 資料集成對關係圖 Pairplot', y=1.02, fontsize=14)
plt.show()

print("觀察重點 Key Observations:")
print("1. petal_length 與 petal_width 高度正相關 (highly correlated)")
print("2. setosa 在所有特徵上都明顯分離 (clearly separated)")
print("3. versicolor 與 virginica 在某些特徵上有重疊 (overlap)")

### 2.2 heatmap — 熱力圖（相關係數矩陣 Correlation Matrix）

In [ ]:
# 計算 Titanic 數值欄位的相關係數
numerical_cols = titanic.select_dtypes(include='number')
corr_matrix = numerical_cols.corr()

# 建立上三角遮罩（避免重複顯示）
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5,
            cbar_kws={'label': '相關係數 Correlation Coefficient'})
plt.title('Titanic 數值特徵相關係數矩陣\nNumerical Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

# 找出相關性最強的變數對
print("\n相關性最強的變數對 (排除對角線):")
corr_pairs = corr_matrix.unstack()
corr_pairs = corr_pairs[corr_pairs != 1.0]  # 排除自身
print(corr_pairs.abs().sort_values(ascending=False).head(6))

### 2.3 violinplot + boxplot — 分布比較

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 小提琴圖：年齡分布（依艙等與存活）
sns.violinplot(data=titanic, x='pclass', y='age', hue='survived',
               split=True, inner='quart', palette={0: '#E53935', 1: '#43A047'},
               ax=axes[0])
axes[0].set_title('年齡分布（依艙等與存活）\nAge by Class & Survival')
axes[0].set_xlabel('艙等 Class')
axes[0].set_ylabel('年齡 Age')
axes[0].legend(title='存活 Survived', labels=['罹難 0', '存活 1'])

# 箱型圖：票價分布（依艙等）
sns.boxplot(data=titanic, x='pclass', y='fare', palette='Set2', ax=axes[1])
axes[1].set_title('票價分布（依艙等）\nFare by Class')
axes[1].set_xlabel('艙等 Class')
axes[1].set_ylabel('票價 Fare')
axes[1].set_ylim(0, 200)  # 限制 Y 軸以便觀察

# Iris 小提琴圖：花瓣長度分布（依物種）
sns.violinplot(data=iris, x='species', y='petal_length',
               palette='husl', inner='stick', ax=axes[2])
axes[2].set_title('花瓣長度分布（依物種）\nPetal Length by Species')
axes[2].set_xlabel('物種 Species')
axes[2].set_ylabel('花瓣長度 Petal Length (cm)')

plt.tight_layout()
plt.show()

### 2.4 更多 Seaborn 圖表：jointplot, countplot, swarmplot

In [ ]:
# 聯合分布圖 Joint Distribution Plot
g = sns.jointplot(data=iris, x='sepal_length', y='petal_length',
                  hue='species', kind='scatter', palette='husl',
                  height=7, marginal_kws={'fill': True})
g.fig.suptitle('花萼長 vs 花瓣長 聯合分布 Joint Distribution', y=1.02)
plt.show()

In [ ]:
# 分類計數圖 + 蜂群圖組合
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 計數圖：各艙等的存活人數
sns.countplot(data=titanic, x='pclass', hue='survived',
              palette={0: '#E53935', 1: '#43A047'}, ax=axes[0])
axes[0].set_title('各艙等存活人數 Survival Count by Class')
axes[0].set_xlabel('艙等 Class')
axes[0].set_ylabel('人數 Count')
axes[0].legend(title='Survived', labels=['罹難', '存活'])

# 蜂群圖：票價分布的每個資料點
sns.swarmplot(data=titanic.sample(200, random_state=42),  # 抽樣避免過密
              x='pclass', y='fare', hue='survived',
              palette={0: '#E53935', 1: '#43A047'}, size=4, ax=axes[1])
axes[1].set_title('票價蜂群圖（每點=一位乘客）\nFare Swarm Plot')
axes[1].set_xlabel('艙等 Class')
axes[1].set_ylabel('票價 Fare')
axes[1].set_ylim(0, 200)

plt.tight_layout()
plt.show()

---

## 3. Plotly 互動圖表 Plotly Interactive Charts

Plotly 產出的圖表支援懸停 (Hover)、縮放 (Zoom)、篩選 (Filter) 等互動操作。

### 3.1 互動散佈圖 Interactive Scatter Plot

In [ ]:
# Iris 互動散佈圖
fig = px.scatter(iris, x='sepal_length', y='sepal_width',
                 color='species', size='petal_length',
                 hover_data=['petal_width'],
                 title='鳶尾花特徵互動散佈圖 Iris Interactive Scatter Plot',
                 labels={
                     'sepal_length': '花萼長度 Sepal Length (cm)',
                     'sepal_width': '花萼寬度 Sepal Width (cm)',
                     'species': '物種 Species',
                     'petal_length': '花瓣長度 Petal Length (cm)',
                     'petal_width': '花瓣寬度 Petal Width (cm)'
                 },
                 color_discrete_sequence=px.colors.qualitative.Set2)

fig.update_layout(height=500)
fig.show()

print("互動操作提示 Interactive Tips:")
print("- 將滑鼠移到資料點上可看到詳細資訊 (hover)")
print("- 點擊圖例可開關特定物種 (click legend to toggle)")
print("- 拖曳可縮放 (drag to zoom)，雙擊重置 (double-click to reset)")

In [ ]:
# Titanic 互動散佈圖 — 使用 Plotly Express
titanic_plot = titanic.dropna(subset=['age', 'fare']).copy()
titanic_plot['survived_label'] = titanic_plot['survived'].map({0: '罹難 Died', 1: '存活 Survived'})
titanic_plot['class_label'] = titanic_plot['pclass'].map({1: '頭等艙', 2: '二等艙', 3: '三等艙'})

fig = px.scatter(titanic_plot, x='age', y='fare',
                 color='survived_label', symbol='class_label',
                 hover_data=['sex', 'embarked', 'class_label'],
                 title='Titanic 乘客分布：年齡 vs 票價 (Age vs Fare)',
                 labels={'age': '年齡 Age', 'fare': '票價 Fare',
                         'survived_label': '存活狀態', 'class_label': '艙等'},
                 color_discrete_map={'罹難 Died': '#E53935', '存活 Survived': '#43A047'},
                 opacity=0.7)

fig.update_layout(height=500, yaxis_range=[0, 300])
fig.show()

### 3.2 3D 散佈圖 3D Scatter Plot

In [ ]:
# Iris 3D 散佈圖 — 花萼長 x 花萼寬 x 花瓣長
fig = px.scatter_3d(iris, x='sepal_length', y='sepal_width', z='petal_length',
                    color='species', symbol='species',
                    title='鳶尾花 3D 特徵空間 Iris 3D Feature Space',
                    labels={
                        'sepal_length': '花萼長',
                        'sepal_width': '花萼寬',
                        'petal_length': '花瓣長'
                    },
                    color_discrete_sequence=px.colors.qualitative.Set2,
                    opacity=0.8)

fig.update_layout(height=600,
                  scene=dict(
                      xaxis_title='花萼長度 Sepal Length',
                      yaxis_title='花萼寬度 Sepal Width',
                      zaxis_title='花瓣長度 Petal Length'
                  ))
fig.show()

print("3D 操作提示:")
print("- 拖曳旋轉 (drag to rotate)")
print("- 滾輪縮放 (scroll to zoom)")
print("- 可清楚看到 setosa 在 3D 空間中完全分離")

### 3.3 動態更新與多面板 Animation & Subplots

In [ ]:
# 互動直方圖 + 邊際箱型圖 (Marginal Plot)
fig = px.histogram(titanic.dropna(subset=['age']),
                   x='age', color='survived',
                   nbins=30, marginal='box',
                   title='年齡分布（依存活狀態）Age Distribution by Survival',
                   labels={'age': '年齡 Age', 'survived': '存活 Survived'},
                   color_discrete_map={0: '#E53935', 1: '#43A047'},
                   barmode='overlay', opacity=0.7)

fig.update_layout(height=500)
fig.show()

In [ ]:
# Plotly 多面板儀表板 Multi-Panel Dashboard
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=('年齡分布 Age Distribution',
                                    '票價分布 Fare Distribution',
                                    '各艙等存活率 Survival by Class',
                                    '性別存活率 Survival by Gender'),
                    specs=[[{'type': 'histogram'}, {'type': 'histogram'}],
                           [{'type': 'bar'}, {'type': 'bar'}]])

# 年齡直方圖
fig.add_trace(go.Histogram(x=titanic['age'].dropna(), nbinsx=30,
                           marker_color='steelblue', name='Age'), row=1, col=1)

# 票價直方圖
fig.add_trace(go.Histogram(x=titanic['fare'], nbinsx=30,
                           marker_color='coral', name='Fare'), row=1, col=2)

# 各艙等存活率
survival_class = titanic.groupby('pclass')['survived'].mean()
fig.add_trace(go.Bar(x=['1st', '2nd', '3rd'], y=survival_class.values,
                     marker_color=['#2196F3', '#FF9800', '#4CAF50'],
                     name='By Class', text=[f'{v:.1%}' for v in survival_class.values],
                     textposition='auto'), row=2, col=1)

# 性別存活率
survival_sex = titanic.groupby('sex')['survived'].mean()
fig.add_trace(go.Bar(x=survival_sex.index, y=survival_sex.values,
                     marker_color=['#E91E63', '#1565C0'],
                     name='By Gender', text=[f'{v:.1%}' for v in survival_sex.values],
                     textposition='auto'), row=2, col=2)

fig.update_layout(height=700, title_text='Titanic EDA 互動儀表板 Interactive Dashboard',
                  showlegend=False)
fig.show()

---

## 4. 完整 EDA 流程示範 Complete EDA Workflow

以下使用 Titanic 資料集，示範完整的四步驟 EDA 流程。

### Step 1: 資料概覽 Data Overview

In [ ]:
# Step 1: 資料概覽 Data Overview
print("=" * 60)
print("Step 1: 資料概覽 Data Overview")
print("=" * 60)

print(f"\n資料形狀 Shape: {titanic.shape}")
print(f"列數 Rows: {titanic.shape[0]}, 欄數 Columns: {titanic.shape[1]}")

print("\n--- 欄位資訊 Column Info ---")
print(titanic.dtypes)

print("\n--- 描述統計 Descriptive Statistics ---")
display(titanic.describe())

print("\n--- 類別欄位統計 Categorical Statistics ---")
display(titanic.describe(include='object'))

print("\n--- 缺失值摘要 Missing Value Summary ---")
missing = pd.DataFrame({
    '缺失數量': titanic.isnull().sum(),
    '缺失比例(%)': (titanic.isnull().mean() * 100).round(2)
})
display(missing[missing['缺失數量'] > 0].sort_values('缺失比例(%)', ascending=False))

### Step 2: 單變數分析 Univariate Analysis

In [ ]:
# Step 2: 單變數分析 — 數值變數
print("=" * 60)
print("Step 2: 單變數分析 Univariate Analysis")
print("=" * 60)

num_cols = ['age', 'fare', 'sibsp', 'parch']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, col in enumerate(num_cols):
    row, col_idx = divmod(i, 2)
    ax = axes[row, col_idx]
    
    # 直方圖 + KDE
    sns.histplot(titanic[col].dropna(), kde=True, ax=ax, color='steelblue', alpha=0.7)
    
    # 標記均值與中位數
    mean_val = titanic[col].mean()
    median_val = titanic[col].median()
    ax.axvline(mean_val, color='red', linestyle='--', linewidth=1.5, label=f'Mean={mean_val:.2f}')
    ax.axvline(median_val, color='green', linestyle=':', linewidth=1.5, label=f'Median={median_val:.2f}')
    
    ax.set_title(f'{col} 分布 Distribution')
    ax.legend(fontsize=9)

plt.suptitle('數值變數分布 Numerical Variable Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 偏態與峰態
print("\n偏態 (Skewness) 與峰態 (Kurtosis):")
for col in num_cols:
    skew = titanic[col].skew()
    kurt = titanic[col].kurtosis()
    skew_type = '右偏 Right-skewed' if skew > 0.5 else ('左偏 Left-skewed' if skew < -0.5 else '近似對稱 Symmetric')
    print(f"  {col}: Skewness={skew:.3f} ({skew_type}), Kurtosis={kurt:.3f}")

In [ ]:
# Step 2 續: 單變數分析 — 類別變數
cat_cols = ['survived', 'pclass', 'sex', 'embarked']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for i, col in enumerate(cat_cols):
    value_counts = titanic[col].value_counts()
    sns.barplot(x=value_counts.index.astype(str), y=value_counts.values,
               palette='Set2', ax=axes[i])
    axes[i].set_title(f'{col} 分布')
    axes[i].set_ylabel('人數 Count')
    
    # 加上百分比標記
    total = len(titanic)
    for j, v in enumerate(value_counts.values):
        axes[i].text(j, v + 3, f'{v}\n({v/total*100:.1f}%)', ha='center', fontsize=9)

plt.suptitle('類別變數分布 Categorical Variable Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Step 3: 雙變數/多變數分析 Bivariate/Multivariate Analysis

In [ ]:
# Step 3a: 相關性分析 Correlation Analysis
print("=" * 60)
print("Step 3: 雙變數/多變數分析 Bivariate/Multivariate Analysis")
print("=" * 60)

# 相關係數熱力圖
corr = titanic[['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']].corr()

plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title('特徵相關係數矩陣 Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

print("\n關鍵發現:")
print("- fare 與 survived 正相關 (0.26): 票價越高，存活率越高")
print("- pclass 與 survived 負相關 (-0.34): 艙等數字越小(越高級)，存活率越高")
print("- pclass 與 fare 負相關 (-0.55): 高級艙等票價更高（合理）")

In [ ]:
# Step 3b: 存活率 vs 各因素 Survival vs Factors
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 性別 vs 存活率
sns.barplot(data=titanic, x='sex', y='survived', palette=['#E91E63', '#1565C0'], ax=axes[0, 0])
axes[0, 0].set_title('性別 vs 存活率 Gender vs Survival Rate')
axes[0, 0].set_ylabel('存活率 Survival Rate')

# 艙等 vs 存活率
sns.barplot(data=titanic, x='pclass', y='survived', palette='Set2', ax=axes[0, 1])
axes[0, 1].set_title('艙等 vs 存活率 Class vs Survival Rate')
axes[0, 1].set_ylabel('存活率 Survival Rate')

# 年齡分布 — 依存活狀態
sns.kdeplot(data=titanic[titanic['survived'] == 0], x='age', fill=True,
            alpha=0.5, color='#E53935', label='罹難 Died', ax=axes[1, 0])
sns.kdeplot(data=titanic[titanic['survived'] == 1], x='age', fill=True,
            alpha=0.5, color='#43A047', label='存活 Survived', ax=axes[1, 0])
axes[1, 0].set_title('年齡密度（依存活）Age KDE by Survival')
axes[1, 0].legend()

# 票價 vs 存活 — 箱型圖
sns.boxplot(data=titanic, x='survived', y='fare',
            palette={0: '#E53935', 1: '#43A047'}, ax=axes[1, 1])
axes[1, 1].set_title('票價 vs 存活 Fare vs Survival')
axes[1, 1].set_ylim(0, 200)
axes[1, 1].set_xticklabels(['罹難 Died', '存活 Survived'])

plt.suptitle('存活因素分析 Survival Factor Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Step 3c: 交叉表分析 Cross-tabulation

# 性別 x 艙等 x 存活率
ct = pd.crosstab([titanic['sex'], titanic['pclass']], titanic['survived'], normalize='index')
print("性別 x 艙等 存活率 Survival Rate by Gender & Class:")
display(ct.round(3))

# 視覺化
ct_plot = titanic.groupby(['sex', 'pclass'])['survived'].mean().unstack()
ct_plot.plot(kind='bar', figsize=(10, 5), color=['#2196F3', '#FF9800', '#4CAF50'])
plt.title('存活率：性別 x 艙等 Survival Rate by Gender x Class')
plt.ylabel('存活率 Survival Rate')
plt.xlabel('性別 Gender')
plt.legend(title='艙等 Class')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("\n重要發現: 頭等艙女性存活率最高 (96.8%)，三等艙男性存活率最低 (13.5%)")

### Step 4: 缺失值分析與處理 Missing Value Analysis & Handling

In [ ]:
# Step 4a: 缺失值視覺化
print("=" * 60)
print("Step 4: 缺失值分析 Missing Value Analysis")
print("=" * 60)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 缺失值矩陣
sns.heatmap(titanic.isnull(), cbar=True, yticklabels=False, cmap='YlOrRd', ax=axes[0])
axes[0].set_title('缺失值矩陣 Missing Value Matrix')
axes[0].set_xlabel('欄位 Columns')

# 缺失值比例長條圖
missing_pct = (titanic.isnull().mean() * 100).sort_values(ascending=True)
missing_pct = missing_pct[missing_pct > 0]
colors = ['#FFC107' if v < 10 else '#FF5722' if v < 50 else '#B71C1C' for v in missing_pct.values]
missing_pct.plot(kind='barh', color=colors, ax=axes[1])
axes[1].set_title('各欄位缺失比例 Missing Rate by Column')
axes[1].set_xlabel('缺失比例 Missing Rate (%)')
for i, v in enumerate(missing_pct.values):
    axes[1].text(v + 0.5, i, f'{v:.1f}%', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n缺失值分析結論:")
print("- deck 欺失比例過高 (77.2%)，可考慮刪除該欄位")
print("- age 缺失約 19.9%，需選擇合適的填補策略")
print("- embarked 缺失極少 (0.2%)，可用眾數填補")
print("- embark_town 缺失極少 (0.2%)，與 embarked 相同")

In [ ]:
# Step 4b: 缺失值處理 — 以 age 欄位為例

# 分析 age 的缺失機制：缺失是否與其他變數有關？
print("age 缺失 vs 非缺失的比較:")
age_missing = titanic.copy()
age_missing['age_missing'] = titanic['age'].isnull().astype(int)

# 比較缺失與非缺失組的其他特徵
comparison = age_missing.groupby('age_missing').agg({
    'pclass': 'mean',
    'fare': 'mean',
    'survived': 'mean',
    'sibsp': 'mean'
}).round(3)
comparison.index = ['有 age', '缺 age']
display(comparison)
print("-> age 缺失者的平均艙等較高(pclass更大)，fare較低，可能為 MAR")

# 方法 1：全體中位數填補
age_median_all = titanic['age'].median()
titanic_m1 = titanic.copy()
titanic_m1['age_filled'] = titanic_m1['age'].fillna(age_median_all)

# 方法 2：分組中位數填補（依艙等與性別）
titanic_m2 = titanic.copy()
titanic_m2['age_filled'] = titanic_m2.groupby(['pclass', 'sex'])['age'].transform(
    lambda x: x.fillna(x.median())
)

# 比較填補前後的分布
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 原始分布
sns.histplot(titanic['age'].dropna(), kde=True, color='steelblue', alpha=0.7, ax=axes[0])
axes[0].set_title(f'原始分布 Original (n={titanic["age"].notna().sum()})')
axes[0].set_xlabel('年齡 Age')

# 方法 1：全體中位數
sns.histplot(titanic_m1['age_filled'], kde=True, color='coral', alpha=0.5, ax=axes[1], label='填補後')
sns.histplot(titanic['age'].dropna(), kde=True, color='steelblue', alpha=0.3, ax=axes[1], label='原始')
axes[1].axvline(age_median_all, color='red', linestyle='--', label=f'填補值={age_median_all:.0f}')
axes[1].set_title('方法1: 全體中位數填補 Overall Median')
axes[1].legend(fontsize=9)

# 方法 2：分組中位數
sns.histplot(titanic_m2['age_filled'], kde=True, color='#43A047', alpha=0.5, ax=axes[2], label='填補後')
sns.histplot(titanic['age'].dropna(), kde=True, color='steelblue', alpha=0.3, ax=axes[2], label='原始')
axes[2].set_title('方法2: 分組中位數填補 Grouped Median')
axes[2].legend(fontsize=9)

plt.suptitle('缺失值填補前後分布比較 Imputation Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n比較結論:")
print(f"  原始 age 均值: {titanic['age'].mean():.2f}, 標準差: {titanic['age'].std():.2f}")
print(f"  方法1 均值: {titanic_m1['age_filled'].mean():.2f}, 標準差: {titanic_m1['age_filled'].std():.2f}")
print(f"  方法2 均值: {titanic_m2['age_filled'].mean():.2f}, 標準差: {titanic_m2['age_filled'].std():.2f}")
print("  方法2（分組填補）較好地保留了原始分布的形狀")

### Step 5: 異常值偵測 Outlier Detection

In [ ]:
# 異常值偵測：以 fare 欄位為例
print("=" * 60)
print("異常值偵測 Outlier Detection (fare)")
print("=" * 60)

# IQR 方法
Q1 = titanic['fare'].quantile(0.25)
Q3 = titanic['fare'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = titanic[(titanic['fare'] < lower_bound) | (titanic['fare'] > upper_bound)]
print(f"\nIQR 方法:")
print(f"  Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}")
print(f"  下界={lower_bound:.2f}, 上界={upper_bound:.2f}")
print(f"  異常值數量: {len(outliers)} ({len(outliers)/len(titanic)*100:.1f}%)")

# 視覺化
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 箱型圖
sns.boxplot(y=titanic['fare'], color='steelblue', ax=axes[0])
axes[0].set_title('票價箱型圖 Fare Box Plot')
axes[0].set_ylabel('票價 Fare')

# 直方圖標記異常值區域
sns.histplot(titanic['fare'], bins=50, color='steelblue', ax=axes[1])
axes[1].axvline(upper_bound, color='red', linestyle='--', label=f'上界 Upper={upper_bound:.0f}')
axes[1].set_title('票價直方圖（紅線=異常值邊界）')
axes[1].legend()

# 散佈圖：標記異常值
normal = titanic[titanic['fare'] <= upper_bound]
axes[2].scatter(range(len(normal)), normal['fare'], alpha=0.4, s=10, color='steelblue', label='正常值')
axes[2].scatter(outliers.index, outliers['fare'], alpha=0.7, s=30, color='red', label='異常值', marker='x')
axes[2].axhline(upper_bound, color='red', linestyle='--', alpha=0.5)
axes[2].set_title('資料點分布（紅x=異常值）')
axes[2].set_xlabel('Index')
axes[2].set_ylabel('票價 Fare')
axes[2].legend()

plt.suptitle('票價異常值偵測 Fare Outlier Detection', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### EDA 洞察摘要 Insights Summary

In [ ]:
print("=" * 60)
print("EDA 洞察摘要 Insights Summary")
print("=" * 60)

insights = [
    "1. 性別是最強的存活預測因子：女性存活率 (74.2%) 遠高於男性 (18.9%)",
    "2. 艙等與存活高度相關：頭等艙存活率 63.0%，三等艙僅 24.2%",
    "3. 票價與存活正相關 (r=0.26)：高票價乘客更可能存活",
    "4. 年齡缺失約 19.9%，缺失可能與艙等有關 (MAR)，建議分組填補",
    "5. 票價分布嚴重右偏 (Skewness=4.79)，有極端高價異常值",
    "6. 頭等艙女性存活率高達 96.8%，三等艙男性僅 13.5%：性別與艙等有交互效果",
    "7. deck 欄位 77.2% 缺失，建議刪除或轉為 has_deck 二元特徵"
]

for insight in insights:
    print(f"  {insight}")

print("\n建模建議 Modeling Suggestions:")
print("  - 重要特徵: sex, pclass, fare, age")
print("  - 需處理: age 缺失值, fare 異常值, deck 高缺失欄位")
print("  - 可嘗試: sex * pclass 交互特徵")

---

## 5. 練習題 Exercises

### 練習 1：Seaborn 統計圖表（基礎）

使用 `tips` 資料集（`sns.load_dataset('tips')`），完成以下任務：

1. 繪製 `total_bill` 的直方圖（含 KDE），並用 `hue='time'` 分組（午餐 vs 晚餐）
2. 繪製 `total_bill` vs `tip` 的散佈圖，用 `hue='smoker'` 區分
3. 繪製相關係數熱力圖

In [ ]:
# 練習 1：請在此撰寫你的程式碼
tips = sns.load_dataset('tips')
print(tips.head())
print(tips.shape)

# TODO: 1. 繪製 total_bill 分組直方圖


# TODO: 2. 繪製 total_bill vs tip 散佈圖


# TODO: 3. 繪製相關係數熱力圖


### 練習 2：Plotly 互動圖表（基礎）

使用 `tips` 資料集，完成以下任務：

1. 用 `px.scatter` 繪製 `total_bill` vs `tip` 的互動散佈圖，`color='day'`, `size='size'`
2. 用 `px.box` 繪製各天的小費分布箱型圖

In [ ]:
# 練習 2：請在此撰寫你的程式碼

# TODO: 1. Plotly 互動散佈圖


# TODO: 2. Plotly 互動箱型圖


### 練習 3：EDA 實作（進階）

使用 `penguins` 資料集（`sns.load_dataset('penguins')`），完成以下 EDA 步驟：

1. **資料概覽：** 顯示 shape, info, describe, 缺失值摘要
2. **單變數分析：** 繪製 `bill_length_mm` 和 `body_mass_g` 的分布圖
3. **多變數分析：** 繪製 pairplot（hue='species'）和相關係數熱力圖
4. **洞察：** 列出至少 2 個發現

In [ ]:
# 練習 3：請在此撰寫你的程式碼
penguins = sns.load_dataset('penguins')

# TODO: Step 1 - 資料概覽


# TODO: Step 2 - 單變數分析


# TODO: Step 3 - 多變數分析


# TODO: Step 4 - 洞察


### 挑戰題：進階缺失值處理（選做）

使用 `penguins` 資料集，完成以下任務：

1. 找出所有缺失值，繪製缺失值矩陣
2. 分析 `bill_length_mm` 的缺失是否與 `species` 有關（判斷 MCAR vs MAR）
3. 使用 `sklearn.impute.KNNImputer` 填補數值欄位的缺失值
4. 比較 KNN 填補 vs 中位數填補的分布差異

**提示：**
```python
from sklearn.impute import KNNImputer
imputer = KNNImputer(n_neighbors=5)
df_imputed = pd.DataFrame(imputer.fit_transform(df_numerical), columns=df_numerical.columns)
```

In [ ]:
# 挑戰題：請在此撰寫你的程式碼

# TODO: 1. 缺失值矩陣


# TODO: 2. 缺失機制分析


# TODO: 3. KNN 填補


# TODO: 4. 比較填補方法


---

## 課堂筆記 Class Notes

### 本週重點回顧
- **Matplotlib：** 使用 OO API (`fig, ax = plt.subplots()`)，GridSpec 建立不等大小子圖
- **Seaborn：** pairplot (成對關係), heatmap (相關矩陣), violinplot (分布比較)
- **Plotly：** 互動散佈圖 (hover, zoom, filter), 3D 散佈圖, make_subplots
- **EDA 四步驟：** 概覽 → 單變數 → 多變數 → 洞察
- **缺失值：** MCAR/MAR/MNAR 三種機制，分組填補通常優於全體填補
- **異常值：** IQR 法偵測，需依業務意義決定保留或處理

### 下週預告
**第 3 週：監督式學習概念、資料分割與交叉驗證**
- 訓練集/測試集分割 (Train/Test Split)
- k 折交叉驗證 (k-Fold Cross-Validation)
- 過擬合 (Overfitting) 與欠擬合 (Underfitting)